# 04 — OpenAI Structured Outputs API — Practical Examples

**Covers**: `.parse()` API, raw JSON schema, refusal handling, streaming, compatibility, production pattern

In [ ]:
import os, json
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Optional, Literal
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o-mini'
print('✅ Setup complete')

## 1. `.parse()` vs `.create()` — Side by Side

In [ ]:
from pydantic import BaseModel

class WeatherInfo(BaseModel):
    city: str
    temperature_c: float
    condition: Literal["sunny", "cloudy", "rainy", "snowy", "windy"]
    humidity_pct: int
    recommendation: str

input_text = "What's the weather like in Tokyo? It's 14°C, cloudy, 68% humidity."

# --- Option A: .create() with raw JSON schema ---
raw_schema = WeatherInfo.model_json_schema()
response_a = client.chat.completions.create(
    model=MODEL,
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "weather_info",
            "strict": True,
            "schema": raw_schema
        }
    },
    messages=[{"role": "user", "content": input_text}]
)
raw_content = response_a.choices[0].message.content
data_a = json.loads(raw_content)  # Manual parse required
print(f"Option A (.create): {data_a}")

# --- Option B: .parse() with Pydantic ---
response_b = client.beta.chat.completions.parse(
    model=MODEL,
    response_format=WeatherInfo,           # Direct Pydantic class
    messages=[{"role": "user", "content": input_text}]
)
data_b = response_b.choices[0].message.parsed   # Already a WeatherInfo object
print(f"Option B (.parse):  city={data_b.city}, temp={data_b.temperature_c}°C, {data_b.condition}")
print(f"Recommendation: {data_b.recommendation}")

## 2. Strict Mode Guarantees — Testing the Boundaries

In [ ]:
# Test strict mode with a schema that has specific Literal choices
class MoodClassifier(BaseModel):
    mood: Literal["happy", "sad", "angry", "neutral", "excited"]
    intensity: Literal["low", "medium", "high"]
    is_positive: bool

texts_to_classify = [
    "I just got promoted! Best day ever!",
    "Traffic is terrible and I'm going to be late again.",
    "The meeting has been rescheduled to 3pm.",
    "My team won the championship!"
]

print(f"{'Text':<45} {'Mood':<10} {'Intensity':<10} {'Positive'}")
print("-" * 75)
for text in texts_to_classify:
    r = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "user", "content": f"Classify mood: {text}"}],
        response_format=MoodClassifier
    )
    m = r.choices[0].message.parsed
    # mood is ALWAYS one of the 5 valid Literals — guaranteed!
    print(f"{text[:43]:<45} {m.mood:<10} {m.intensity:<10} {m.is_positive}")

## 3. Refusal Handling — The Safety Mechanism

In [ ]:
class SafeExtraction(BaseModel):
    content: str
    category: str

def safe_extract(text: str) -> SafeExtraction | None:
    """Extract with refusal handling."""
    response = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "user", "content": text}],
        response_format=SafeExtraction
    )
    msg = response.choices[0].message
    
    # ALWAYS check refusal before accessing parsed
    if msg.refusal:
        print(f"⚠️  Model refused: {msg.refusal[:100]}")
        return None
    
    return msg.parsed

# Test with normal content
result1 = safe_extract("This is a friendly message about cooking pasta.")
if result1:
    print(f"✅ Extracted: category={result1.category!r}")

# Test with edge case content (model may or may not refuse)
result2 = safe_extract("Summarize: report on illegal drug manufacturing techniques.")
if result2:
    print(f"ℹ️  Got structured response anyway: {result2.category!r}")
else:
    print("🚫 Model refused this request")

## 4. Multi-Level Nested Schemas

In [ ]:
class Author(BaseModel):
    name: str
    nationality: Optional[str] = None

class Chapter(BaseModel):
    number: int
    title: str
    page_start: int
    page_end: int

class BookCatalogEntry(BaseModel):
    title: str
    author: Author
    isbn: Optional[str] = None
    year: int
    genre: Literal["fiction", "non-fiction", "science", "biography", "self-help", "tech"]
    language: str = "English"
    key_chapters: list[Chapter]
    rating: float = Field(ge=0.0, le=5.0)
    available_formats: list[Literal["hardcover", "paperback", "ebook", "audiobook"]]

book_description = """
'Clean Code' by Robert C. Martin (American), ISBN 978-0-13-235088-4, published 2008.
A technology book about software craftsmanship. Rated 4.4/5.
Available as paperback and ebook. Key chapters: 1. Clean Code (p.1-22), 
2. Meaningful Names (p.23-42), 3. Functions (p.43-75), 4. Comments (p.76-99).
"""

r = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[{"role": "user", "content": f"Create catalog entry for:\n{book_description}"}],
    response_format=BookCatalogEntry
)
book = r.choices[0].message.parsed
rprint(book.model_dump())
print(f"\nAuthor: {book.author.name} ({book.author.nationality})")
print(f"Chapters: {len(book.key_chapters)}, from p{book.key_chapters[0].page_start} to p{book.key_chapters[-1].page_end}")
print(f"Formats: {book.available_formats}")

## 5. Token Usage Analysis

In [ ]:
# Compare token usage: unstructured vs JSON mode vs strict structured
test_prompt = {"role": "user", "content": "Extract: name='Alice', age=30, job='engineer', city='NYC'"}

class SimpleExtract(BaseModel):
    name: str
    age: int
    job: str
    city: str

# Unstructured
r1 = client.chat.completions.create(model=MODEL, messages=[test_prompt])
u1 = r1.usage

# JSON Mode
r2 = client.chat.completions.create(
    model=MODEL,
    response_format={"type": "json_object"},
    messages=[{"role": "system", "content": "Return JSON"}, test_prompt]
)
u2 = r2.usage

# Structured Output
r3 = client.beta.chat.completions.parse(
    model=MODEL,
    response_format=SimpleExtract,
    messages=[test_prompt]
)
u3 = r3.usage

print(f"{'Mode':<20} {'Prompt':<10} {'Completion':<12} {'Total'}")
print("-" * 50)
print(f"{'Unstructured':<20} {u1.prompt_tokens:<10} {u1.completion_tokens:<12} {u1.total_tokens}")
print(f"{'JSON Mode':<20} {u2.prompt_tokens:<10} {u2.completion_tokens:<12} {u2.total_tokens}")
print(f"{'Strict Struct.':<20} {u3.prompt_tokens:<10} {u3.completion_tokens:<12} {u3.total_tokens}")
print(f"\nSchema overhead: ~{u3.prompt_tokens - u1.prompt_tokens} extra prompt tokens for schema")

## 6. Production Pattern — Complete Error-Handled Extractor

In [ ]:
from dataclasses import dataclass
from openai import OpenAIError
import time

@dataclass
class ExtractionResult:
    success: bool
    data: object | None
    error: str | None
    latency_ms: float

class InvoiceData(BaseModel):
    vendor_name: str
    invoice_number: str
    total_amount: float
    currency: str = "USD"
    line_items: list[str]
    due_date: Optional[str] = None
    is_paid: bool = False

def extract_invoice(text: str, model: str = MODEL) -> ExtractionResult:
    """Production-grade invoice extraction with full error handling."""
    start = time.perf_counter()
    try:
        response = client.beta.chat.completions.parse(
            model=model,
            messages=[
                {"role": "system", "content": "You are an invoice data extractor."},
                {"role": "user",   "content": f"Extract invoice data:\n{text}"}
            ],
            response_format=InvoiceData,
            temperature=0.0
        )
        latency = (time.perf_counter() - start) * 1000
        msg = response.choices[0].message
        if msg.refusal:
            return ExtractionResult(False, None, f"Refused: {msg.refusal}", latency)
        return ExtractionResult(True, msg.parsed, None, latency)
    except OpenAIError as e:
        latency = (time.perf_counter() - start) * 1000
        return ExtractionResult(False, None, f"API error: {e}", latency)
    except Exception as e:
        latency = (time.perf_counter() - start) * 1000
        return ExtractionResult(False, None, f"Unexpected: {e}", latency)

invoice_text = """
INVOICE #INV-2024-0892
From: CloudServices Ltd.
Total: $3,450.00 (USD)
Items: Cloud Hosting - $2,000, Support Package - $1,000, Setup Fee - $450
Due: January 15, 2025
Status: Unpaid
"""

result = extract_invoice(invoice_text)
if result.success:
    inv = result.data
    print(f"✅ Extracted in {result.latency_ms:.0f}ms")
    print(f"   Vendor:  {inv.vendor_name}")
    print(f"   Invoice: {inv.invoice_number}")
    print(f"   Total:   ${inv.total_amount:,.2f} {inv.currency}")
    print(f"   Items:   {inv.line_items}")
    print(f"   Due:     {inv.due_date}")
else:
    print(f"❌ Failed: {result.error}")

## 📌 Summary

- **`.parse()`** is the recommended API — typed return, no JSON string parsing
- **`.create()` with `json_schema`** — when you need raw dict output
- **`msg.refusal`** → always check before `msg.parsed`
- **Strict mode guarantees**: all required fields, no extra fields, correct types
- **Semantic correctness NOT guaranteed** — only structural correctness
- **Token overhead**: ~20-100 extra prompt tokens for schema; usually worth it